In [ ]:
from dotenv import load_dotenv,find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
   raise ValueError('GEMINI_API_KEY is not set in the env file')
print('GEMINI_API_KEY loaded successfully')    
llm = ChatGoogleGenerativeAI(api_key=GEMINI_API_KEY,model='gemini-1.5-flash')

In [ ]:
from IPython.display import Markdown
Markdown((llm.invoke('hi').content))

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph,START,END
from IPython.display import Markdown
class GraphState(TypedDict):
    story_theme: str
    generated_story: str
    
def generate_story(state:GraphState)->GraphState:
    story_theme = state['story_theme']
    generate_story_prompt ="""
    Imagine you are a skilled storyteller tasked with creating a simple and interesting story based on the theme below.  
  
    Theme: {theme}
    Your goal is to generate a detailed and realistic story for 10 years old children
    The story should have:
    - A clear beginning, middle, and end.
    - Easy-to-understand language.
    - Detail and meaningful sentences.
    - Interesting characters and events.
    
    Story:
    """
    

    sys_msg = SystemMessage(content="You are a storyteller who writes clear and engaging stories in simple English.")
    hum_msg =  HumanMessage(content=generate_story_prompt.format(theme=story_theme))
    resp = llm.invoke([sys_msg,hum_msg])
    state['generated_story'] = resp.content
    return state 


workflow = StateGraph(GraphState)
workflow.add_node('generate_story',generate_story)
workflow.add_edge(START,'generate_story')
workflow.add_edge('generate_story',END)
app = workflow.compile()
story_theme =  "Once, a rabbit mocked a slow-moving tortoise. The tortoise challenged him to a race. Confident of his speed, the rabbit dashed ahead and took a nap. Meanwhile, the tortoise kept moving steadily. By the time the rabbit woke up, the tortoise had already crossed the finish line."
resp1 = app.invoke({'story_theme':story_theme})
 
Markdown(resp1['generated_story'])

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing import Sequence
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from IPython.display import display, Markdown
import time


class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    object_description: str
    save_audio_path: str
    save_image_path: str
    audio_duration: str
    save_video_path: str


class GraphState(TypedDict):
    story_theme: str
    generated_story:str
    scene_list: list[ScenesList]
    supporting_characters: list[SupportingCharacters]
    main_characters: list[MainCharacters]
    pre_processing_video_path: str
    combined_audio_path: str
    voices_folder: str
    images_folder: str
    videos_folder: str
    messages: Annotated[Sequence[BaseMessage], add_messages]


class SubState(TypedDict):
    current_scene: ScenesList
    output_folder: str


def generate_story_char(state: GraphState) -> GraphState:
    story = state["generated_story"]
    new_prompt = """Based on the Story

    {story},

    create a brief description of the main and supporting character, object, or scene. Include specific details about appearance, characteristics . This description will be used to maintain consistency across multiple scenes.
    and also generate a narration that exactly follows this text, starting with 'Once upon a time...' and keeping all sentences unchanged. Do not change the wording or style.
    Extract structured JSON data from this story: 
    
    Return only valid JSON (without markdown formatting). Do NOT wrap it in triple backticks (```json). Ensure it follows this structure:

    {{
      "main_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
            "supporting_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
          "scenes": [
            {{
              "id": "1",
              "scene": "Engaging Beginning",
              "description": "Begin with a captivating moment to grab children's attention.",
              "narration": ""
              "object_description": ""
            }},
         
          ],   
    }}

    Strictly output **only JSON** without extra text.
    """

    sys_msg = SystemMessage(
        content="You are an assistant that extracts all character names and details from a story."
    )
    hum_msg = HumanMessage(content=new_prompt.format(story=story))
    print(f"Before generating the story characters")
    start_time = time.time()
    res = llm.invoke([sys_msg, hum_msg])
    end_time = time.time()
    print(f"Time taken to generate story characters: {end_time - start_time} seconds")
    print(f"After generating the story characters:{res}")
    parsed_resp = json.loads(res.content)
    print(f"Parsed response: {parsed_resp}")
    for scene in parsed_resp["scenes"]:

        prompt_template = f"""Create a detailed, photorealistic image of the following scene 
        {scene["description"]} 

        **Mood & Lighting**: Cinematic, immersive atmosphere, realistic lighting to match the scene's emotions.

        Ensure character consistency throughout all images. The illustration should capture the story’s essence and atmosphere.
        """
        scene["img_prompt"] = prompt_template

    state["scene_list"] = parsed_resp["scenes"] 
    state["supporting_characters"] = parsed_resp["supporting_characters"]
    state["main_characters"] = parsed_resp["main_characters"]
    return state


workflow = StateGraph(GraphState)
workflow.add_node("generate_story_char", generate_story_char)
workflow.add_edge(START, "generate_story_char")
workflow.add_edge("generate_story_char", END)
app = workflow.compile()

resp2 = app.invoke({"generated_story": resp1["generated_story"]})

for scene in resp2["scene_list"]:
    display(Markdown(f"\n{'='*40}"))
    display(Markdown(f"### Scene {scene['id']}: {scene['scene']}"))
    display(Markdown(f"---"))
    display(Markdown(f"**📖 Description:** {scene['description']}"))
    display(Markdown(f"\n**📜 Narration:**\n{scene['narration']}"))
    display(Markdown(f"\n**🖼️ Image Prompt:**\n{scene['img_prompt']}"))
    display(Markdown(f"{'='*40}\n"))

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from datetime import datetime
import os
import nest_asyncio
import time
from gtts import gTTS
from typing import Sequence
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from langgraph.graph.message import add_messages
from typing import Any
from langgraph.types import Send
import asyncio
import edge_tts
import pyttsx3
from moviepy import *
nest_asyncio.apply()


class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    object_description: str
    save_audio_path: str
    save_image_path: str
    audio_duration: str
    save_video_path: str


class GraphState(TypedDict):
    story_theme: str
    generated_story:str
    scene_list: list[ScenesList]
    supporting_characters: list[SupportingCharacters]
    main_characters: list[MainCharacters]
    pre_processing_video_path: str
    combined_audio_path: str
    voices_folder: str
    images_folder: str
    videos_folder: str
    messages: Annotated[Sequence[BaseMessage], add_messages]


class SubState(TypedDict):
    current_scene: ScenesList
    output_folder: str


async def save_voice(save_path, current_voice):
    await current_voice.save(save_path)


async def call_llm_gen_voice(voice):
    try:
        tts = edge_tts.Communicate(
            voice, voice="en-US-JennyNeural", volume="+100%", pitch="+5Hz"
        )
        return tts
    except Exception as e:
        raise ValueError("Error while generating the voice", e)


async def generate_scene_voice(state: SubState):
    cur_scene = state["current_scene"]
    scene_id = int(cur_scene["id"])
    output_folder = state["output_folder"]
    narration_text = cur_scene.get("narration", "")

    try:
        print(f"Generating voice for scene {scene_id}")
        generated_voice = await call_llm_gen_voice(narration_text)
        save_path = os.path.join(output_folder, f"voice_scene_{scene_id}.mp3")
        save_time = time.time()

        await save_voice(save_path, generated_voice)
        end_save_time = time.time()
        
        print(f"saving time for this scene {scene_id}={end_save_time-save_time}")
        print(f"Generated voice successfully for scene {scene_id}")
        try:
            audio_duration = AudioFileClip(save_path).duration
        except Exception as e:
            print(f"Error generating voice for scene {scene_id}: {e}")  
        message = HumanMessage(
            content="",
            additional_kwargs={
                "voice_id": scene_id,
                "save_path": save_path,
                "audio_duration": audio_duration
            }
        )
        
        return {"messages": [message]}

    except Exception as e:
        print(f"Error generating voice for scene {scene_id}: {e}")


def continue_generate_voice(state: GraphState):
    scene_list = state["scene_list"]
    time_stamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"voices_{time_stamp}"
    output_folder = os.path.join("New_Generated_voices", dynamic_folder)
    output_folder = os.path.join("arman_output", output_folder)
    os.makedirs(output_folder, exist_ok=True)
    return [
        Send(
            "generate_scene_voice",
            {
                "current_scene": scene,
                "output_folder": output_folder,
            },
        )
        for scene in scene_list
    ]


def final_node(state: GraphState):
    total_messages = state["messages"]
    print(f"Toatal messages:{total_messages}")
    scene_list = state["scene_list"]
    list_voice_folder = set()
    for scene in scene_list:
        get_voiceid = int(scene["id"])
        for msg in total_messages:

            if get_voiceid == int(msg.additional_kwargs.get("voice_id")):
                print(f"{get_voiceid} {msg}")
                scene["save_audio_path"] = msg.additional_kwargs.get("save_path", "")
                scene["audio_duration"] = msg.additional_kwargs.get("audio_duration", "")
                list_voice_folder.add(os.path.dirname(msg.additional_kwargs.get("save_path", "")))
                
                break
    state["voices_folder"] = list(list_voice_folder)[0] if list_voice_folder else "" 
    print(f'voices folder :{list(list_voice_folder)[0] if list_voice_folder else "" }')
    return state
      


def sync_generate_scene_voice(state: SubState):
    return asyncio.run(generate_scene_voice(state))


workflow = StateGraph(GraphState)
# workflow.add_node("generate_scene_voice", generate_scene_voice)
workflow.add_node("generate_scene_voice", sync_generate_scene_voice)
workflow.add_node("final_node", final_node)

workflow.add_conditional_edges(START, continue_generate_voice, ["generate_scene_voice"])
workflow.add_edge("generate_scene_voice", "final_node")
workflow.add_edge("final_node", END)

app = workflow.compile()


async def run_workflow():
    response = await app.ainvoke({"scene_list": resp2["scene_list"]})
    
    print(response)
    return response

loop = asyncio.get_event_loop()

respe_data=loop.run_until_complete(run_workflow())

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from datetime import datetime
import os
from gtts import gTTS 
from typing import Sequence
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from langgraph.graph.message import add_messages
from typing import Any
import logging
from langgraph.types import Send
import requests
import random


class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    object_description: str
    save_audio_path: str
    save_image_path: str
    audio_duration: str
    save_video_path: str


class GraphState(TypedDict):
    story_theme: str
    generated_story:str
    scene_list: list[ScenesList]
    supporting_characters: list[SupportingCharacters]
    main_characters: list[MainCharacters]
    pre_processing_video_path: str
    combined_audio_path: str
    voices_folder: str
    images_folder: str
    videos_folder: str
    messages: Annotated[Sequence[BaseMessage], add_messages]


class SubState(TypedDict):
    current_scene: ScenesList
    output_folder: str



def saveImage(image_content, save_path):
    output_folder = os.path.dirname(save_path)
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    with open(save_path, "wb") as f:
        f.write(image_content)


def images_generates(prompt: str):
    try: 
        color_value1 = format(random.randint(0, 16777215), '06x')  
        color_value2 = format(random.randint(0, 16777215), '06x')
        image_url = f"https://placehold.co/1080x1920/{color_value1}/{color_value2}/png" 
        return image_url 

    except Exception as e:
        logging.error(f"Unexpected error generating image: {e}")
        raise


 

def generate_scence_image(state:SubState):
    cur_scene = state["current_scene"]
    scene_id = int(cur_scene["id"])
    output_folder = state["output_folder"]  
    image_prompt = cur_scene.get("Img_prompt", "")
    try:
        print(f"Generating image for scene {scene_id}")
        imageUrl = images_generates(image_prompt)
        if not imageUrl:
            raise ValueError("Generated image URL is empty.")
        imageContent = requests.get(imageUrl).content
        if not imageContent:
            raise ValueError("Failed to retrieve image content.")
        save_path = os.path.join(output_folder, f"image_scene_{scene_id}.png")
        saveImage(imageContent, save_path)
        print(f"Images saved to {save_path}")
        message = HumanMessage(content='',additional_kwargs={"image_id": scene_id,"save_path": save_path})
        return {'messages':[message]}

    except Exception as e:
        logging.error(f"Error processing scene {scene_id}: {e}")
       
def final_node(state:GraphState):
    total_messages = state['messages'] 
    scene_list = state['scene_list'] 
    image_list =set()
    for scene in scene_list:
        get_imageid = scene['id'] 
        for msg in total_messages:
            if int(get_imageid)== msg.additional_kwargs.get('image_id'): 
                scene['save_image_path'] = msg.additional_kwargs.get('save_path','')
                image_list.add(os.path.dirname(msg.additional_kwargs.get('save_path','')))
                break        
    state['images_folder']= list(image_list)[0] if image_list else ""  
    
    return state      
    # return {'Scene_list':scene_list}
     

def continue_scence_image(state:GraphState):
    scene_list = state['scene_list']
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"images_{timestamp}"
    output_folder = os.path.join("New_Generated_images", dynamic_folder)
    output_folder = os.path.join("arman_output", output_folder)
    os.makedirs(output_folder, exist_ok=True) 
    return [Send("generate_scence_image",{'current_scene':value, 'output_folder':output_folder}) for value in scene_list ]
     
     
     
     
     
     
workflow = StateGraph(GraphState)
workflow.add_node('generate_scence_image',generate_scence_image)
workflow.add_node('final_node',final_node)
 

workflow.add_conditional_edges(START,continue_scence_image,['generate_scence_image'])
workflow.add_edge('generate_scence_image','final_node')
workflow.add_edge('final_node',END)
app = workflow.compile()

resp3 = app.invoke({"scene_list": resp2['scene_list']})
print(resp3)

In [ ]:

import cv2
import numpy as np
import os
from datetime import datetime
from moviepy import *
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing import Sequence
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from IPython.display import display, Markdown
from PIL import Image, ImageDraw, ImageFont
from pydub import AudioSegment
import math
class ClassObject(TypedDict):
    object: str
    description: str

class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str

class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str

class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    save_audio_path: str
    save_image_path: str
    save_video_path:str
    save_audio_video_path: str
    audio_duration: str
    objects: list[ClassObject]

class GraphState(TypedDict): 
    main_characters: list[MainCharacters]
    supporting_characters: list[SupportingCharacters]
    scene_list: list[ScenesList]
    messages: Annotated[Sequence[BaseMessage], add_messages]
    combine_audio_video_key: Annotated[Sequence[BaseMessage], add_messages]
    videos_with_audio_folder: str
    pre_processing_video_path: str
    voices_folder: str
    images_folder: str
    videos_folder:str
    videos_with_audio_folder:str
    

class SubState(TypedDict): 
    current_scene: ScenesList
    output_folder: str
 
 
def zoom_in(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-in effect."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + (i / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames

def zoom_out(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-out effect without blinking."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + ((num_frames - i - 1) / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames

def fade_in(image, num_frames=2):
    """Creates a fade-in effect from black to the image."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames

def fade_out(image, num_frames=30):
    """Creates a fade-out effect from image to black."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = 1 - i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames

# ----------------------- Pre-processing Video Function -----------------------
def pre_processing_video(state: SubState):
    cur_scene = state['current_scene']
    scene_id = int(cur_scene['id'])
    output_folder = state['output_folder']
    fps = 10    

    first_img = cv2.imread(cur_scene["save_image_path"])
    if first_img is None:
        raise ValueError(f"Cannot load image: {cur_scene['save_image_path']}")
    
    h, w, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    pre_processing_video_path = os.path.join(output_folder, f"video_saving_{scene_id}.mp4")
    video_writer = cv2.VideoWriter(pre_processing_video_path, fourcc, fps, (w, h)) 
    
    img_path = cur_scene["save_image_path"]
    duration = float(cur_scene["audio_duration"]) 
    total_frames = math.ceil(duration * fps)

    fade_duration = int(fps * 1)
    core_frames = total_frames - 2 * fade_duration if total_frames > 2 * fade_duration else max(1, total_frames - fade_duration)

    img = cv2.imread(img_path) 
    img = cv2.resize(img, (w, h))

    if scene_id == 0:
        zoom_in_frames = zoom_in(img, num_frames=core_frames, zoom_factor=0.5)
        fade_out_frames = fade_out(zoom_in_frames[-1], num_frames=fade_duration)
        effect_frames = zoom_in_frames + fade_out_frames
    elif scene_id % 2 == 1:
        zoom_out_frames = zoom_out(img, num_frames=core_frames, zoom_factor=0.5)
        fade_out_frames = fade_out(zoom_out_frames[-1], num_frames=fade_duration)
        effect_frames = zoom_out_frames + fade_out_frames
    else:
        fade_in_frames = fade_in(img, num_frames=fade_duration)
        zoom_in_frames = zoom_in(img, num_frames=core_frames, zoom_factor=0.5)
        fade_out_frames = fade_out(zoom_in_frames[-1], num_frames=fade_duration)
        effect_frames = fade_in_frames + zoom_in_frames + fade_out_frames

    # Ensure frame count matches exactly
    if len(effect_frames) > total_frames:
        effect_frames = effect_frames[:total_frames]
    elif len(effect_frames) < total_frames:
        last_frame = effect_frames[-1]
        effect_frames += [last_frame] * (total_frames - len(effect_frames))

    for frame in effect_frames:
        video_writer.write(frame)
    
    video_writer.release()
    print(f"✅ Audio Duration for scene {scene_id}: {duration:.2f} sec")
    
    final_video = VideoFileClip(pre_processing_video_path)
    final_video_duration = final_video.duration
    print(f"✅ Final Video Duration for scene {scene_id}: {final_video_duration:.2f} sec (No Audio)")
    print(f"Pre-processing video saved as {pre_processing_video_path}")
    
    message = HumanMessage(content='', additional_kwargs={"video_id": scene_id, "save_path": pre_processing_video_path})
    return {'messages': [message]}

def save_paths(state: GraphState):
    total_messages = state["messages"]
    print(f"Toatal messages:{total_messages}")
    scene_list = state["scene_list"]
    video_list =set()
    for scene in scene_list:
        get_id = int(scene["id"])
     
        for msg in total_messages:
            video_id = msg.additional_kwargs.get('video_id') 
            
            if video_id is not None and get_id == video_id: 
                save_path = msg.additional_kwargs.get('save_path','')
                print(f"{get_id} {msg}")
                scene["save_video_path"] = save_path 
                video_list.add(os.path.dirname(save_path)) 
    state['videos_folder'] =list(video_list)[0]  if video_list else ""          
    return state

def continue_generate_videos(state: GraphState):
    scene_list = state['scene_list']
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"videolist_{timestamp}"
    output_folder = os.path.join("video_list", dynamic_folder)
    output_folder = os.path.join("arman_output", output_folder)
    os.makedirs(output_folder, exist_ok=True) 
    return [Send('pre_processing_video', {'current_scene':value,'output_folder':output_folder}) for value in scene_list]

def combine_audio_with_video(state: SubState):
    cur_scene = state['current_scene']
    scene_id = int(cur_scene['id'])
    video_path = cur_scene['save_video_path']
    audio_path = cur_scene['save_audio_path']
    output_folder = state['output_folder']
 
    
    combining_audio_with_video_path = os.path.join(output_folder, f"video_with_audio_{scene_id}.mp4")
    !ffmpeg -i "{video_path}" -i "{audio_path}" -c:v copy -c:a aac -strict experimental "{combining_audio_with_video_path}"
    print(f'audio_path==================={audio_path}')
    print(f'output_folder==================={output_folder}')
    message = HumanMessage(content='', additional_kwargs={"scene_id": scene_id, "save_path": combining_audio_with_video_path})
    return {"combine_audio_video_key":[message]}
def final_node(state: GraphState):
    total_messages = state["combine_audio_video_key"]
    print(f"Toatal messages:{total_messages}")
    scene_list = state["scene_list"]
    video_list =set()
    for scene in scene_list:
        get_id = int(scene["id"])
     
        for msg in total_messages:
            scene_id = msg.additional_kwargs.get('scene_id') 
            
            if scene_id is not None and get_id == scene_id: 
                save_path = msg.additional_kwargs.get('save_path','')
                print(f"{get_id} {msg}")
                scene["save_audio_video_path"] = save_path 
                video_list.add(os.path.dirname(save_path)) 
    state['videos_with_audio_folder'] =list(video_list)[0]  if video_list else ""          
    return state
def combining_videosddd(state:GraphState):
    scene_list = state['scene_list'] 
    sorted_scene = sorted(scene_list, key=lambda x: int(x['id']))
    video_files = [scene['save_audio_video_path'] for scene in sorted_scene]
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S") 
    output_folder = os.path.join("arman_output", 'combined_videos') 
    os.makedirs(output_folder, exist_ok=True)
    concat_txt_path = "arman_output/concat_list.txt"
    with open(concat_txt_path, "w") as f:
        for file in video_files:
            f.write(f"file '{os.path.abspath(file)}'\n")
    import subprocess
    final_output_video = os.path.join(output_folder, f"combined_video_{timestamp}.mp4")
    command = [
    "ffmpeg",
    "-f", "concat",
    "-safe", "0",
    "-i", concat_txt_path,
    "-c", "copy",
    final_output_video
]

    

    process = subprocess.run(command, capture_output=True, text=True)
    state['pre_processing_video_path'] = final_output_video
    print("FFmpeg stdout:", process.stdout)
    print("FFmpeg stderr:", process.stderr)

  
def merge_single_audiovideo(state:GraphState):
    scene_list = state['scene_list'] 
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = os.path.join('videos_with_audio', f"video_with_audio_{timestamp}")
    output_folder = os.path.join("arman_output", dynamic_folder)
    os.makedirs(output_folder, exist_ok=True)
    return [Send('combine_audio_with_video', {'current_scene':value,'output_folder':output_folder}) for value in scene_list]

workflow = StateGraph(GraphState)
workflow.add_node('pre_processing_video', pre_processing_video)
workflow.add_node('combine_audio_with_video', combine_audio_with_video)
workflow.add_node('save_paths', save_paths)
workflow.add_node('final_node', final_node)
workflow.add_node('combining_videos', combining_videos) 


workflow.add_conditional_edges(START,continue_generate_videos,['pre_processing_video'])
workflow.add_edge('pre_processing_video','save_paths')
workflow.add_conditional_edges('save_paths',merge_single_audiovideo,['combine_audio_with_video'])
workflow.add_edge('combine_audio_with_video','final_node')
workflow.add_edge('final_node','combining_videos') 
workflow.add_edge('combining_videos',END)
 
app = workflow.compile() 
resp4 = app.invoke({"scene_list": resp3['scene_list']})
print(resp4)
